# Unsupervised validation of anomaly detection methods

Goal

Develop a robust unsupervised ranking methodology for binuclear Fe-containing complexes.

Candidate structures are NOT used during model development and will be evaluated only after the methodology has been finalized.

The methodology will be compared using:

- Positive hold-out validation
- Robustness to random seed
- Robustness to train/test split
- Robustness across feature representations
- Robustness across anomaly detection algorithms

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

np.random.seed(42)

## Data loading and curation

Run from the `notebooks/` directory. Paths are relative to the repository root (`../data`, `../results`).


In [ ]:
features = pd.read_csv(
    "../data/raw/binuclear_fe_with_smiles.csv"
)

emb = pd.read_parquet(
    "../data/chemberta_embeddings.parquet"
)

print(features.shape)
print(emb.shape)

(990, 190)
(970, 770)


In [ ]:
df = features.merge(
    emb,
    on="cod_id",
    how="inner",
    validate="one_to_one"
)

print(df.shape)

(970, 959)


In [ ]:
df['cod_norm'] = (df['cod_id'].astype(str)
                  .str.replace(r'^COD_?', '', regex=True)
                  .str.lstrip('0'))
print(len(df), df['cod_norm'].nunique())

970 834


In [ ]:
metals = ["Mn","Co","Ni","Cu","Zn","Ba","Mo"]

metal_group = []

for _, row in df.iterrows():

    label = "Fe-Fe"

    for m in metals:
        if row[f"has_{m}"] == 1:
            label = f"Fe-{m}"

    metal_group.append(label)

df["metal_group"] = metal_group

print(df["metal_group"].value_counts())

metal_group
Fe-Fe    658
Fe-Co     78
Fe-Ni     74
Fe-Cu     57
Fe-Mn     52
Fe-Zn     29
Fe-Mo     18
Fe-Ba      4
Name: count, dtype: int64


In [ ]:
df['cod_norm'] = (df['cod_id'].astype(str)
                  .str.replace(r'^COD_?', '', regex=True)
                  .str.lstrip('0'))
print(len(df), df['cod_norm'].nunique())
print(df.groupby('metal_group')['cod_norm'].nunique())

970 834
metal_group
Fe-Ba      4
Fe-Co     54
Fe-Cu     42
Fe-Fe    618
Fe-Mn     37
Fe-Mo     14
Fe-Ni     46
Fe-Zn     19
Name: cod_norm, dtype: int64


In [ ]:
# Normalized COD identifier
df["cod_norm"] = (
    df["cod_id"]
      .astype(str)
      .str.replace(r"^COD_?", "", regex=True)
      .str.lstrip("0")
)

# Drop duplicate depositions by cod_norm
df_clean = (
    df
    .drop_duplicates(subset="cod_norm", keep="first")
    .reset_index(drop=True)
)

# Check
print(len(df_clean), df_clean["cod_norm"].nunique())
print(df_clean.groupby("metal_group")["cod_norm"].nunique())

# Save
df_clean.to_csv("combined_dataset_clean.csv", index=False)

834 834
metal_group
Fe-Ba      4
Fe-Co     54
Fe-Cu     42
Fe-Fe    618
Fe-Mn     37
Fe-Mo     14
Fe-Ni     46
Fe-Zn     19
Name: cod_norm, dtype: int64


> **Reproducing from this repository:** cells above rebuild `data/data_train.csv` from the raw COD extract (`data/raw/binuclear_fe_with_smiles.csv`), which is not distributed here. If you only have the files in `data/`, start from the next cell.


In [ ]:
df = pd.read_csv("../data/data_train.csv")  # curated 834-compound reference set

In [ ]:
df.shape

(834, 960)

In [ ]:
exclude = {
    "cod_id",
    "source",
    "type",
    "formula_sum",
    "smiles",
    "smiles;",
    "metal_group",
}

phys_cols = [
    c
    for c in df.select_dtypes(include=np.number).columns
    if c not in exclude
    and not c.startswith("emb_")
]

print(len(phys_cols))

953


In [ ]:
emb_cols = [
    c
    for c in df.columns
    if c.startswith("chemberta_")
]

print(len(emb_cols))

768


In [ ]:
exclude = {
    "cod_id",
    "source",
    "type",
    "formula_sum",
    "smiles",
    "smiles;",
    "metal_group",

    # must be excluded
    "label",
    "target",
    "cod_norm",
}

phys_cols = [
    c
    for c in df.select_dtypes(include=np.number).columns
    if (
        c not in exclude
        and not c.startswith("chemberta_")
    )
]

print("Number of PhysChem features:", len(phys_cols))

X_phys = df[phys_cols].values

print("PhysChem matrix:", X_phys.shape)

Number of PhysChem features: 183
PhysChem matrix: (834, 183)


In [ ]:
combined_cols = phys_cols + emb_cols

print(len(combined_cols))

951


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

def evaluate_model(
    X,
    groups,
    model,
    n_splits=4,
    random_state=42,
    use_scaler=True,
    use_imputer=True,
):
    """
    Positive hold-out evaluation.

    Parameters
    ----------
    X : ndarray
        Feature matrix.
    groups : array-like
        Stratification labels.
    model : sklearn estimator
        Unsupervised anomaly detector.
    """

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    scores = np.zeros(len(X))
    folds = np.zeros(len(X), dtype=int)

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, groups)):

        steps = []

        if use_imputer:
            steps.append(
                ("imputer", SimpleImputer(strategy="median"))
            )

        if use_scaler:
            steps.append(
                ("scaler", StandardScaler())
            )

        pipe = Pipeline(
            steps + [("model", model)]
        )

        pipe.fit(X[train_idx])

        scores[test_idx] = pipe.decision_function(
            X[test_idx]
        )

        folds[test_idx] = fold

    result = pd.DataFrame({
    "cod_id": df["cod_id"].values,
    "formula_sum": df["formula_sum"].values,
    "metal": df["metal_group"].values,
    "fold": folds,
    "score": scores,
})

    return result

In [ ]:
X_phys = df[phys_cols].to_numpy(dtype=float)

X_emb = df[emb_cols].to_numpy(dtype=float)

X_combined = np.hstack([X_phys, X_emb])

groups = df["metal_group"].values

print(X_phys.shape)
print(X_emb.shape)
print(X_combined.shape)

(834, 183)
(834, 768)
(834, 951)


# **PhysChem**

# **baseline  PhysChem + Isolation Forest**

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=500,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

In [ ]:
res_phys = evaluate_model(
    X=X_phys,
    groups=groups,
    model=model,
)

In [ ]:
res_phys["cod_id"] = df["cod_id"].values
res_phys["formula_sum"] = df["formula_sum"].values
res_phys["metal"] = df["metal_group"].values

In [ ]:
print(res_phys.score.describe())

count    834.000000
mean       0.080728
std        0.058589
min       -0.211921
25%        0.056976
50%        0.096834
75%        0.124456
max        0.147550
Name: score, dtype: float64


In [ ]:
res_phys.sort_values(
    "score"
).head(20)

,cod_id,formula_sum,metal,fold,score
499,COD4322041,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe,3,-0.211921
782,COD7225179,C10 H57 Fe2 La N6 O89 P2 W20,Fe-Fe,3,-0.206377
588,COD7004034,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe,3,-0.204444
784,COD7225180,C12 H97 Fe2 N8 O105 P2 Sm W20,Fe-Fe,3,-0.203802
783,COD7225178,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe,1,-0.202383
668,COD7037888,C12 H60 Fe2 N8 Na2 O88 Sb2 W20,Fe-Fe,3,-0.193719
785,COD7225181,C12 H89 Fe2 N8 O101 P2 Tb W20,Fe-Fe,2,-0.163521
498,COD4322043,C20 H107 Bi2 Fe2 N5 O92 W20,Fe-Fe,3,-0.152378
111,7063786,C6 H24 Cu Fe I5 N6 Pb,Fe-Cu,0,-0.116573
809,COD7703149,C20 H32 Cu5 Fe2 Gd N22 Na O18,Fe-Cu,1,-0.106117


In [ ]:
res_phys.sort_values(
    "score",
    ascending=False
).head(20)

,cod_id,formula_sum,metal,fold,score
728,COD7102855,C12 H14 Fe2 N2 O5 S2,Fe-Fe,1,0.147550
695,COD7054179,C15 H17 Fe2 N O6 S2,Fe-Fe,2,0.147137
699,COD7054178,C14 H15 Fe2 N O6 S2,Fe-Fe,1,0.147132
242,COD4061803,C16 H24 Fe2 N2 O6 S2,Fe-Fe,1,0.146743
290,COD4072970,C17 H23 Fe2 N3 O5.25 S2,Fe-Fe,2,0.146661
284,COD4072698,C15 H16 Fe2 N2 O6 S2,Fe-Fe,0,0.146613
283,COD4072699,C15 H16 Fe2 N2 O6 S2,Fe-Fe,2,0.146482
250,COD4062908,C16 H24 Fe2 N2 O6 S2,Fe-Fe,0,0.145801
287,COD4072972,C16 H18 Fe2 N4 O4 S2,Fe-Fe,3,0.143451
288,COD4072973,C17 H20 Fe2 N4 O4 S2,Fe-Fe,3,0.143312


## **Random seed robustness**

In [ ]:
from sklearn.ensemble import IsolationForest

seeds = [
    1,
    5,
    10,
    21,
    42,
    77,
    100,
    777,
    2025,
    9999,
]

seed_scores = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in seeds:

    model = IsolationForest(
        n_estimators=500,
        contamination="auto",
        random_state=seed,
        n_jobs=-1,
    )

    res = evaluate_model(
        X=X_phys,
        groups=groups,
        model=model,
        random_state=seed,
    )

    seed_scores[f"seed_{seed}"] = res["score"].values

print(seed_scores.shape)

seed_scores.head()

(834, 11)


,cod_id,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
0,1515817,0.004058,0.008067,0.015794,0.007952,0.005032,0.010712,0.011167,0.011892,0.013378,0.009780
1,1515819,-0.056925,-0.056212,-0.058656,-0.057160,-0.052803,-0.063155,-0.062127,-0.056684,-0.054665,-0.038972
2,1515818,-0.016333,-0.023163,-0.016111,-0.021015,-0.036025,-0.037581,-0.028864,-0.023553,-0.015950,-0.027173
3,1515820,-0.046333,-0.041096,-0.054779,-0.048621,-0.045397,-0.056741,-0.048237,-0.042014,-0.044172,-0.050879
4,1519184,-0.004435,-0.001786,-0.004974,-0.004737,-0.005532,-0.017701,-0.028706,-0.005978,-0.012366,-0.009134


**Rank correlation across random seeds**

In [ ]:
from scipy.stats import spearmanr

score_cols = [c for c in seed_scores.columns if c.startswith("seed_")]

corr = pd.DataFrame(
    index=score_cols,
    columns=score_cols,
    dtype=float,
)

for c1 in score_cols:
    for c2 in score_cols:
        corr.loc[c1, c2] = spearmanr(
            seed_scores[c1],
            seed_scores[c2]
        ).correlation

print("=" * 60)
print(f"Mean Spearman correlation: {corr.values[np.triu_indices_from(corr,1)].mean():.4f}")
print(f"Minimum correlation      : {corr.values[np.triu_indices_from(corr,1)].min():.4f}")
print(f"Maximum correlation      : {corr.values[np.triu_indices_from(corr,1)].max():.4f}")

display(corr.round(3))

Mean Spearman correlation: 0.9897
Minimum correlation      : 0.9880
Maximum correlation      : 0.9917


,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000,0.991,0.992,0.991,0.990,0.991,0.990,0.990,0.990,0.989
seed_5,0.991,1.000,0.991,0.991,0.989,0.991,0.991,0.989,0.989,0.989
seed_10,0.992,0.991,1.000,0.991,0.990,0.990,0.992,0.990,0.989,0.989
seed_21,0.991,0.991,0.991,1.000,0.990,0.991,0.991,0.988,0.990,0.989
seed_42,0.990,0.989,0.990,0.990,1.000,0.989,0.991,0.988,0.989,0.989
seed_77,0.991,0.991,0.990,0.991,0.989,1.000,0.990,0.988,0.990,0.988
seed_100,0.990,0.991,0.992,0.991,0.991,0.990,1.000,0.990,0.989,0.989
seed_777,0.990,0.989,0.990,0.988,0.988,0.988,0.990,1.000,0.989,0.989
seed_2025,0.990,0.989,0.989,0.990,0.989,0.990,0.989,0.989,1.000,0.988
seed_9999,0.989,0.989,0.989,0.989,0.989,0.988,0.989,0.989,0.988,1.000


**Top-k overlap across random seeds**

In [ ]:
from itertools import combinations

def top_overlap(df, k):

    cols = [c for c in df.columns if c.startswith("seed_")]

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(
            df.nsmallest(k, c1).index
        )

        top2 = set(
            df.nsmallest(k, c2).index
        )

        overlaps.append(
            len(top1 & top2) / k
        )

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10,25,50,100]:

    mean_, min_, max_ = top_overlap(
        seed_scores,
        k
    )

    print("="*60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.944
Min overlap  : 0.800
Max overlap  : 1.000
Top-25
Mean overlap : 0.928
Min overlap  : 0.840
Max overlap  : 1.000
Top-50
Mean overlap : 0.909
Min overlap  : 0.860
Max overlap  : 0.960
Top-100
Mean overlap : 0.920
Min overlap  : 0.890
Max overlap  : 0.960


# **Local Outlier Factor (LOF) PHYS**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

model = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
)

res_lof = evaluate_model(
    X=X_phys,
    groups=groups,
    model=model,
)

res_lof["cod_id"] = df["cod_id"].values
res_lof["formula_sum"] = df["formula_sum"].values
res_lof["metal"] = df["metal_group"].values

print(res_lof.score.describe())

display(
    res_lof.sort_values("score").head(20)
)

display(
    res_lof.sort_values(
        "score",
        ascending=False
    ).head(20)
)

count    834.000000
mean       0.160725
std        0.622168
min      -13.092226
25%        0.045171
50%        0.326635
75%        0.463953
max        0.554753
Name: score, dtype: float64


,cod_id,formula_sum,metal,fold,score
499,COD4322041,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe,3,-13.092226
426,COD4124584,C18 H15 Ca Fe2 N5 O8,Fe-Fe,3,-2.413263
783,COD7225178,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe,1,-1.748183
689,COD7050419,C16 H10 Fe2 Mo O6 S2,Fe-Mo,2,-1.680392
832,1548887,C26 H27 Ba F6 Fe N3 O12.5 S2,Fe-Ba,2,-1.651208
831,1548885,C26 H27 Ba Cl F6 Fe N3 O12 S2,Fe-Ba,0,-1.637157
128,7704674,C15 H13 Fe Mo O6 P S2,Fe-Mo,2,-1.593885
833,1514389,C50 H78 Ba F3 Fe N4 O17 S4,Fe-Ba,3,-1.578963
682,COD7043514,C13 H5 Fe2 K O7 S,Fe-Fe,2,-1.483854
453,COD4304826,C4 H8 Cl2 Fe2 Li2 O13,Fe-Fe,1,-1.382854


,cod_id,formula_sum,metal,fold,score
392,COD4112954,C14 H18 Fe2 N3 O5 P S2,Fe-Fe,3,0.554753
258,COD4064894,C17 H13 Fe2 N O8 S2,Fe-Fe,3,0.541706
589,COD7005027,C13 H9 Fe2 N O7 S2,Fe-Fe,1,0.537848
289,COD4072988,C16 H22 Fe2 N2 O4 S3,Fe-Fe,2,0.536797
242,COD4061803,C16 H24 Fe2 N2 O6 S2,Fe-Fe,1,0.535945
494,COD4319032,C11 H16 Fe2 N O4 P S2,Fe-Fe,2,0.535162
290,COD4072970,C17 H23 Fe2 N3 O5.25 S2,Fe-Fe,2,0.534903
250,COD4062908,C16 H24 Fe2 N2 O6 S2,Fe-Fe,0,0.534566
695,COD7054179,C15 H17 Fe2 N O6 S2,Fe-Fe,2,0.531541
400,COD4114598,C14 H9 Fe2 N O6 S2,Fe-Fe,1,0.531203


## **Random seed robustness (LOF)**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

seeds = [
    1,5,10,21,42,77,100,777,2025,9999
]

seed_scores_lof = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in seeds:

    model = LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
    )

    res = evaluate_model(
        X=X_phys,
        groups=groups,
        model=model,
        random_state=seed,
    )

    seed_scores_lof[f"seed_{seed}"] = res["score"].values

print(seed_scores_lof.shape)

(834, 11)


## **Spearman correlation (LOF)**

In [ ]:
from scipy.stats import spearmanr

score_cols = list(seed_scores_lof.columns[1:])

corr = pd.DataFrame(
    index=score_cols,
    columns=score_cols,
    dtype=float,
)

for c1 in score_cols:
    for c2 in score_cols:
        corr.loc[c1, c2] = spearmanr(
            seed_scores_lof[c1],
            seed_scores_lof[c2]
        ).correlation

vals = corr.values[np.triu_indices(len(score_cols), k=1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

display(corr.round(3))

Mean Spearman correlation: 0.9743
Minimum correlation      : 0.9685
Maximum correlation      : 0.9791


,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000,0.976,0.976,0.979,0.976,0.974,0.976,0.972,0.974,0.977
seed_5,0.976,1.000,0.978,0.979,0.970,0.974,0.974,0.973,0.975,0.973
seed_10,0.976,0.978,1.000,0.979,0.972,0.978,0.974,0.974,0.974,0.974
seed_21,0.979,0.979,0.979,1.000,0.975,0.976,0.977,0.978,0.976,0.977
seed_42,0.976,0.970,0.972,0.975,1.000,0.968,0.973,0.970,0.970,0.974
seed_77,0.974,0.974,0.978,0.976,0.968,1.000,0.973,0.972,0.970,0.973
seed_100,0.976,0.974,0.974,0.977,0.973,0.973,1.000,0.972,0.972,0.975
seed_777,0.972,0.973,0.974,0.978,0.970,0.972,0.972,1.000,0.972,0.973
seed_2025,0.974,0.975,0.974,0.976,0.970,0.970,0.972,0.972,1.000,0.974
seed_9999,0.977,0.973,0.974,0.977,0.974,0.973,0.975,0.973,0.974,1.000


## **Top-k overlap (LOF)**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nsmallest(k, c1).index)
        top2 = set(df_scores.nsmallest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(seed_scores_lof, k)

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.762
Min overlap  : 0.500
Max overlap  : 1.000
Top-25
Mean overlap : 0.791
Min overlap  : 0.680
Max overlap  : 0.880
Top-50
Mean overlap : 0.808
Min overlap  : 0.740
Max overlap  : 0.860
Top-100
Mean overlap : 0.885
Min overlap  : 0.850
Max overlap  : 0.930


# **One-Class SVM PHYS**

In [ ]:
from sklearn.svm import OneClassSVM

model = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05,
)

res_ocsvm = evaluate_model(
    X=X_phys,
    groups=groups,
    model=model,
)

res_ocsvm["cod_id"] = df["cod_id"].values
res_ocsvm["formula_sum"] = df["formula_sum"].values
res_ocsvm["metal"] = df["metal_group"].values

print(res_ocsvm.score.describe())

display(
    res_ocsvm.sort_values("score").head(20)
)

display(
    res_ocsvm.sort_values(
        "score",
        ascending=False
    ).head(20)
)

count    834.000000
mean       0.304115
std        0.307759
min       -1.025938
25%        0.152598
50%        0.378234
75%        0.534815
max        0.761768
Name: score, dtype: float64


,cod_id,formula_sum,metal,fold,score
783,COD7225178,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe,1,-1.025938
809,COD7703149,C20 H32 Cu5 Fe2 Gd N22 Na O18,Fe-Cu,1,-1.015901
499,COD4322041,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe,3,-1.009715
827,COD9012663,;C H6.82 Al0.08 Ca9.348 Ce0.102 Cl0.29 Fe2.19 ...,Fe-Mn,2,-0.943884
782,COD7225179,C10 H57 Fe2 La N6 O89 P2 W20,Fe-Fe,3,-0.892513
588,COD7004034,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe,3,-0.882165
681,COD7042865,C H Cl F Fe Ni P S,Fe-Ni,0,-0.873815
739,COD7108065,C12 H24 Br8 Fe2 N24 Zn,Fe-Zn,0,-0.868875
668,COD7037888,C12 H60 Fe2 N8 Na2 O88 Sb2 W20,Fe-Fe,3,-0.861245
498,COD4322043,C20 H107 Bi2 Fe2 N5 O92 W20,Fe-Fe,3,-0.847879


,cod_id,formula_sum,metal,fold,score
749,COD7114171,C12 H16 Fe2 N6 O5,Fe-Fe,2,0.761768
283,COD4072699,C15 H16 Fe2 N2 O6 S2,Fe-Fe,2,0.730802
339,COD4083203,C12 H13 Fe2 N O6 S2,Fe-Fe,2,0.719919
795,COD7236139,C18 H21 Fe2 N2 O14 S2,Fe-Fe,2,0.710583
695,COD7054179,C15 H17 Fe2 N O6 S2,Fe-Fe,2,0.709979
290,COD4072970,C17 H23 Fe2 N3 O5.25 S2,Fe-Fe,2,0.707975
524,COD4337030,C15 H31 Fe2 N5 O11 S,Fe-Fe,3,0.698875
73,7000409,C12 H12 Fe2 N6 O4 S2,Fe-Fe,2,0.697320
766,COD7201699,C11 H15 Fe2 O8 P S2,Fe-Fe,2,0.693828
166,COD1547478,C9 H18 Fe2 N5 O3 S2,Fe-Fe,0,0.691348


# **Hold**-out robustness (One-Class SVM)

In [ ]:
from sklearn.svm import OneClassSVM

split_seeds = [
    1, 5, 10, 21, 42,
    77, 100, 777, 2025, 9999
]

ocsvm_scores = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in split_seeds:

    model = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
    )

    res = evaluate_model(
        X=X_phys,
        groups=groups,
        model=model,
        random_state=seed,
    )

    ocsvm_scores[f"seed_{seed}"] = res["score"].values

print(ocsvm_scores.shape)

(834, 11)


## **Spearman correlation (SVM)**

In [ ]:
from scipy.stats import spearmanr

score_cols = list(ocsvm_scores.columns[1:])

corr = pd.DataFrame(
    index=score_cols,
    columns=score_cols,
    dtype=float,
)

for c1 in score_cols:
    for c2 in score_cols:
        corr.loc[c1, c2] = spearmanr(
            ocsvm_scores[c1],
            ocsvm_scores[c2]
        ).correlation

vals = corr.values[np.triu_indices(len(score_cols), k=1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

display(corr.round(3))

Mean Spearman correlation: 0.9774
Minimum correlation      : 0.9716
Maximum correlation      : 0.9852


,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000,0.983,0.981,0.978,0.973,0.974,0.973,0.978,0.974,0.978
seed_5,0.983,1.000,0.985,0.981,0.979,0.980,0.979,0.982,0.980,0.984
seed_10,0.981,0.985,1.000,0.978,0.975,0.978,0.978,0.979,0.977,0.983
seed_21,0.978,0.981,0.978,1.000,0.974,0.977,0.972,0.979,0.976,0.980
seed_42,0.973,0.979,0.975,0.974,1.000,0.972,0.972,0.976,0.972,0.977
seed_77,0.974,0.980,0.978,0.977,0.972,1.000,0.972,0.978,0.976,0.977
seed_100,0.973,0.979,0.978,0.972,0.972,0.972,1.000,0.976,0.973,0.977
seed_777,0.978,0.982,0.979,0.979,0.976,0.978,0.976,1.000,0.976,0.981
seed_2025,0.974,0.980,0.977,0.976,0.972,0.976,0.973,0.976,1.000,0.977
seed_9999,0.978,0.984,0.983,0.980,0.977,0.977,0.977,0.981,0.977,1.000


## **Top-k overlap (LOF)**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nsmallest(k, c1).index)
        top2 = set(df_scores.nsmallest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(ocsvm_scores, k)

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.727
Min overlap  : 0.500
Max overlap  : 0.900
Top-25
Mean overlap : 0.844
Min overlap  : 0.760
Max overlap  : 0.920
Top-50
Mean overlap : 0.864
Min overlap  : 0.800
Max overlap  : 0.900
Top-100
Mean overlap : 0.902
Min overlap  : 0.850
Max overlap  : 0.960


# **ChemBERTa embeddings**

## **Baseline: Isolation Forest**

In [ ]:
# ============================================================
# ChemBERTa feature matrix
# ============================================================

emb_cols = [
    c for c in df.columns
    if c.startswith("chemberta_")
]

X_emb = df[emb_cols].values

print(X_emb.shape)
print(f"Number of embedding features: {len(emb_cols)}")
print(f"Missing values: {np.isnan(X_emb).sum()}")

(834, 768)
Number of embedding features: 768
Missing values: 0


In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=500,
    contamination="auto",
    random_state=42,
)

res_if_emb = evaluate_model(
    X=X_emb,
    groups=groups,
    model=model,
)

res_if_emb["cod_id"] = df["cod_id"].values
res_if_emb["formula_sum"] = df["formula_sum"].values
res_if_emb["metal"] = df["metal_group"].values

print(res_if_emb.score.describe())

print("\n20 most anomalous structures\n")
display(
    res_if_emb.sort_values("score").head(20)
)

print("\n20 most typical structures\n")
display(
    res_if_emb.sort_values(
        "score",
        ascending=False
    ).head(20)
)

count    834.000000
mean       0.057462
std        0.037674
min       -0.104572
25%        0.036551
50%        0.061826
75%        0.086966
max        0.118668
Name: score, dtype: float64

20 most anomalous structures



,cod_id,formula_sum,metal,fold,score
576,COD6000335,C6 H4 Fe K Mn N6 O2,Fe-Mn,0,-0.104572
573,COD4518079,C14.5 H8 Cl0 Fe2 N0 O10,Fe-Fe,2,-0.068362
778,COD7224311,C8 H14 Fe2 N O12,Fe-Fe,0,-0.066073
487,COD4317034,C8 H14 Fe2 N O12,Fe-Fe,0,-0.061683
219,COD2235024,C14 H34 Cl6 Fe2 N2 O,Fe-Fe,2,-0.057729
459,COD4305371,C4 H10 Fe2 N4 O4 S2,Fe-Fe,2,-0.057161
666,COD7037769,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe,0,-0.056466
488,COD4317035,C8 H14 Fe Mn N O12,Fe-Mn,2,-0.055531
489,COD4317036,C8 H14 Co Fe N O12,Fe-Co,2,-0.055116
409,COD4118946,C8 H14 Fe2 N O12,Fe-Fe,2,-0.054874



20 most typical structures



,cod_id,formula_sum,metal,fold,score
753,COD7115076,C17 H16 Fe2 O5 S3,Fe-Fe,1,0.118668
549,COD4346025,C12 H14 B F4 Fe2 N3 O5 S2,Fe-Fe,0,0.117326
710,COD7058566,C14 H10 Fe2 O8 S Se,Fe-Fe,1,0.117225
242,COD4061803,C16 H24 Fe2 N2 O6 S2,Fe-Fe,1,0.116915
600,COD7005869,C15 H10 Br Fe2 N O6 S2,Fe-Fe,2,0.116853
720,COD7100042,C15 H10 Br Fe2 N O6 S2,Fe-Fe,0,0.116624
792,COD7232028,C12 H17 Fe2 O5 P S2,Fe-Fe,3,0.116206
294,COD4073173,C16 H13 Fe2 N O8 S2,Fe-Fe,1,0.116206
756,COD7115452,C13 H15 Fe2 O6 P S2,Fe-Fe,1,0.115640
265,COD4067046,C16 H17 Fe2 O7 P S2,Fe-Fe,3,0.115571


### **Hold-out robustness (Isolation Forest)**

In [ ]:
from sklearn.ensemble import IsolationForest

split_seeds = [
    1, 5, 10, 21, 42,
    77, 100, 777, 2025, 9999
]

if_emb_scores = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in split_seeds:

    model = IsolationForest(
        n_estimators=500,
        contamination="auto",
        random_state=42,
    )

    res = evaluate_model(
        X=X_emb,
        groups=groups,
        model=model,
        random_state=seed,
    )

    if_emb_scores[f"seed_{seed}"] = res["score"].values

print(if_emb_scores.shape)

(834, 11)


### **Stability analysis (Isolation Forest)**

In [ ]:
from scipy.stats import spearmanr

score_cols = list(if_emb_scores.columns[1:])

corr = pd.DataFrame(
    index=score_cols,
    columns=score_cols,
    dtype=float,
)

for c1 in score_cols:
    for c2 in score_cols:
        corr.loc[c1, c2] = spearmanr(
            if_emb_scores[c1],
            if_emb_scores[c2]
        ).correlation

vals = corr.values[np.triu_indices(len(score_cols), k=1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

display(corr.round(3))

Mean Spearman correlation: 0.9872
Minimum correlation      : 0.9846
Maximum correlation      : 0.9895


,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000,0.987,0.985,0.987,0.986,0.986,0.985,0.987,0.985,0.986
seed_5,0.987,1.000,0.987,0.989,0.989,0.988,0.988,0.989,0.989,0.989
seed_10,0.985,0.987,1.000,0.985,0.986,0.987,0.985,0.986,0.986,0.985
seed_21,0.987,0.989,0.985,1.000,0.988,0.987,0.986,0.988,0.987,0.988
seed_42,0.986,0.989,0.986,0.988,1.000,0.988,0.986,0.988,0.987,0.988
seed_77,0.986,0.988,0.987,0.987,0.988,1.000,0.988,0.989,0.988,0.989
seed_100,0.985,0.988,0.985,0.986,0.986,0.988,1.000,0.988,0.987,0.987
seed_777,0.987,0.989,0.986,0.988,0.988,0.989,0.988,1.000,0.989,0.988
seed_2025,0.985,0.989,0.986,0.987,0.987,0.988,0.987,0.989,1.000,0.987
seed_9999,0.986,0.989,0.985,0.988,0.988,0.989,0.987,0.988,0.987,1.000


### **Top-k overlap (Isolation Forest)**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nsmallest(k, c1).index)
        top2 = set(df_scores.nsmallest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(if_emb_scores, k)

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.729
Min overlap  : 0.500
Max overlap  : 0.900
Top-25
Mean overlap : 0.862
Min overlap  : 0.760
Max overlap  : 0.920
Top-50
Mean overlap : 0.879
Min overlap  : 0.800
Max overlap  : 0.960
Top-100
Mean overlap : 0.927
Min overlap  : 0.900
Max overlap  : 0.950


## **Local Outlier Factor EMB**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

model = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
)

res_lof_emb = evaluate_model(
    X=X_emb,
    groups=groups,
    model=model,
)

res_lof_emb["cod_id"] = df["cod_id"].values
res_lof_emb["formula_sum"] = df["formula_sum"].values
res_lof_emb["metal"] = df["metal_group"].values

print(res_lof_emb.score.describe())

print("\n20 most anomalous structures\n")
display(
    res_lof_emb.sort_values("score").head(20)
)

print("\n20 most typical structures\n")
display(
    res_lof_emb.sort_values(
        "score",
        ascending=False
    ).head(20)
)

count    834.000000
mean       0.407353
std        0.102684
min       -0.861180
25%        0.358254
50%        0.428188
75%        0.478063
max        0.535214
Name: score, dtype: float64

20 most anomalous structures



,cod_id,formula_sum,metal,fold,score
576,COD6000335,C6 H4 Fe K Mn N6 O2,Fe-Mn,0,-0.861180
219,COD2235024,C14 H34 Cl6 Fe2 N2 O,Fe-Fe,2,-0.281533
348,COD4084253,C18 H10 Fe2 O8 Se2,Fe-Fe,3,-0.033407
458,COD4305186,C14 H20 Cl9 Fe2 N3,Fe-Fe,2,0.060229
427,COD4126163,C12 H39 Fe2 N P2 Si2,Fe-Fe,1,0.068627
666,COD7037769,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe,0,0.072774
688,COD7050433,C12 H16 Fe2 N4 O13,Fe-Fe,1,0.079829
342,COD4084245,C18 H10 Fe2 O8 S2,Fe-Fe,0,0.089732
413,COD4122578,C16 H30 Fe2 O4 P2 S2,Fe-Fe,2,0.091536
513,COD4327447,C14 H27 B F4 Fe2 O4 P2 S3,Fe-Fe,1,0.098863



20 most typical structures



,cod_id,formula_sum,metal,fold,score
353,COD4086203,C12 H4 Cl Fe2 O7 P S,Fe-Fe,1,0.535214
354,COD4086206,C16 H12 Br Fe2 O8 P S,Fe-Fe,2,0.533929
619,COD7011106,C9 H6 Fe2 O7 S2,Fe-Fe,0,0.532898
356,COD4086204,C13 H7 Fe2 O8 P S,Fe-Fe,3,0.530266
350,COD4086200,C14 H9 Fe2 O8 P S,Fe-Fe,2,0.529843
301,COD4073682,C15 H11 Fe2 N O8 S3,Fe-Fe,1,0.528986
305,COD4074503,C14 H16 Fe2 O9 S2,Fe-Fe,3,0.528721
476,COD4313486,C14 H25 Fe2 N2 Na O7 S3,Fe-Fe,0,0.528159
695,COD7054179,C15 H17 Fe2 N O6 S2,Fe-Fe,2,0.528032
437,COD4132744,C16 H13 Fe2 N O6 S2,Fe-Fe,2,0.526060


### **Hold-out robustness (Local Outlier Factor)**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

split_seeds = [
    1, 5, 10, 21, 42,
    77, 100, 777, 2025, 9999
]

lof_emb_scores = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in split_seeds:

    model = LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
    )

    res = evaluate_model(
        X=X_emb,
        groups=groups,
        model=model,
        random_state=seed,
    )

    lof_emb_scores[f"seed_{seed}"] = res["score"].values

print(lof_emb_scores.shape)

(834, 11)


### **Spearman correlation (LOF, ChemBERTa)**

In [ ]:
score_cols = [
    c for c in lof_emb_scores.columns
    if c.startswith("seed_")
]

corr = lof_emb_scores[score_cols].corr(method="spearman")

print("=" * 60)
print(
    f"Mean Spearman correlation: "
    f"{corr.values[np.triu_indices_from(corr,1)].mean():.4f}"
)
print(
    f"Minimum correlation      : "
    f"{corr.values[np.triu_indices_from(corr,1)].min():.4f}"
)
print(
    f"Maximum correlation      : "
    f"{corr.values[np.triu_indices_from(corr,1)].max():.4f}"
)

corr.round(3)

Mean Spearman correlation: 0.9499
Minimum correlation      : 0.9179
Maximum correlation      : 0.9653


,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000,0.953,0.932,0.958,0.961,0.950,0.965,0.935,0.961,0.946
seed_5,0.953,1.000,0.936,0.956,0.958,0.959,0.961,0.957,0.958,0.958
seed_10,0.932,0.936,1.000,0.933,0.941,0.935,0.937,0.918,0.939,0.920
seed_21,0.958,0.956,0.933,1.000,0.963,0.955,0.965,0.937,0.965,0.945
seed_42,0.961,0.958,0.941,0.963,1.000,0.955,0.965,0.939,0.965,0.947
seed_77,0.950,0.959,0.935,0.955,0.955,1.000,0.956,0.949,0.958,0.950
seed_100,0.965,0.961,0.937,0.965,0.965,0.956,1.000,0.943,0.965,0.951
seed_777,0.935,0.957,0.918,0.937,0.939,0.949,0.943,1.000,0.941,0.955
seed_2025,0.961,0.958,0.939,0.965,0.965,0.958,0.965,0.941,1.000,0.950
seed_9999,0.946,0.958,0.920,0.945,0.947,0.950,0.951,0.955,0.950,1.000


### **Top-k overlap (LOF, ChemBERTa)**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nsmallest(k, c1).index)
        top2 = set(df_scores.nsmallest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(
        lof_emb_scores,
        k
    )

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.669
Min overlap  : 0.500
Max overlap  : 0.800
Top-25
Mean overlap : 0.818
Min overlap  : 0.720
Max overlap  : 0.920
Top-50
Mean overlap : 0.784
Min overlap  : 0.700
Max overlap  : 0.840
Top-100
Mean overlap : 0.794
Min overlap  : 0.750
Max overlap  : 0.830


# **One-Class SVM (ChemBERTa)**

In [ ]:
from sklearn.svm import OneClassSVM

model = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05,
)

res_ocsvm_emb = evaluate_model(
    X=X_emb,
    groups=groups,
    model=model,
)

res_ocsvm_emb["cod_id"] = df["cod_id"].values
res_ocsvm_emb["formula_sum"] = df["formula_sum"].values
res_ocsvm_emb["metal"] = df["metal_group"].values

print(res_ocsvm_emb.score.describe())

print("\n20 most anomalous structures\n")
display(
    res_ocsvm_emb.sort_values("score").head(20)
)

print("\n20 most typical structures\n")
display(
    res_ocsvm_emb.sort_values(
        "score",
        ascending=False
    ).head(20)
)

count    834.000000
mean       0.562236
std        0.481720
min       -2.393813
25%        0.244389
50%        0.597221
75%        0.878762
max        1.819505
Name: score, dtype: float64

20 most anomalous structures



,cod_id,formula_sum,metal,fold,score
576,COD6000335,C6 H4 Fe K Mn N6 O2,Fe-Mn,0,-2.393813
219,COD2235024,C14 H34 Cl6 Fe2 N2 O,Fe-Fe,2,-1.570191
666,COD7037769,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe,0,-1.007003
787,COD7228299,C18 H26 Cl4 Fe2,Fe-Fe,0,-0.916533
197,COD2020370,C15.9996 H38.44 Fe2.6666 N15.9996 O19.22 Zn3.9999,Fe-Zn,3,-0.862190
183,COD2002093,C12 H18 Al Cl9 Fe2 N6,Fe-Fe,1,-0.856408
373,COD4102423,C12 H10 Fe2 N4 O4 S2,Fe-Fe,2,-0.790694
458,COD4305186,C14 H20 Cl9 Fe2 N3,Fe-Fe,2,-0.787372
627,COD7015223,C17 H42 Cl4 Fe2 N6 O,Fe-Fe,3,-0.776569
88,7009580,C14 H32 Fe2 N2 O16,Fe-Fe,1,-0.731817



20 most typical structures



,cod_id,formula_sum,metal,fold,score
549,COD4346025,C12 H14 B F4 Fe2 N3 O5 S2,Fe-Fe,0,1.819505
550,COD4346026,C12 H14 Fe2 N4 O4 S2,Fe-Fe,3,1.681579
795,COD7236139,C18 H21 Fe2 N2 O14 S2,Fe-Fe,2,1.581323
295,COD4073238,C19 H27 Fe2 N O3 Si3,Fe-Fe,0,1.575150
42,4112972,C10 H12 Fe O4 P Zn0.5,Fe-Zn,0,1.540513
817,COD7709297,C18 H18 Fe2 N2 O2,Fe-Fe,3,1.534784
312,COD4076128,C14 H10 Fe2 Ga2 I4 O4,Fe-Fe,1,1.514384
435,COD4130755,C15.4 H23.8 B Cl0.8 F4 Fe2 N3 O2 S2,Fe-Fe,2,1.509523
665,COD7037837,C12 H23 Cl Fe2 O8 P2 S3,Fe-Fe,1,1.506043
628,COD7015307,C14 H21 Fe2 I O5 S3,Fe-Fe,3,1.497756


## **Hold-out robustness (One-Class SVM)**

In [ ]:
from sklearn.svm import OneClassSVM

split_seeds = [
    1,5,10,21,42,
    77,100,777,2025,9999
]

ocsvm_emb_scores = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in split_seeds:

    model = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
    )

    res = evaluate_model(
        X=X_emb,
        groups=groups,
        model=model,
        random_state=seed,
    )

    ocsvm_emb_scores[f"seed_{seed}"] = res["score"].values

print(ocsvm_emb_scores.shape)

(834, 11)


## **Spearman correlation (One-Class SVM)**

In [ ]:
score_cols = [
    c for c in ocsvm_emb_scores.columns
    if c.startswith("seed_")
]

corr = ocsvm_emb_scores[score_cols].corr(method="spearman")

vals = corr.values[np.triu_indices_from(corr, 1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

display(corr.round(3))

Mean Spearman correlation: 0.9801
Minimum correlation      : 0.9720
Maximum correlation      : 0.9857


,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000,0.976,0.974,0.978,0.976,0.972,0.976,0.975,0.975,0.972
seed_5,0.976,1.000,0.983,0.985,0.986,0.982,0.984,0.983,0.981,0.980
seed_10,0.974,0.983,1.000,0.983,0.983,0.980,0.983,0.982,0.980,0.979
seed_21,0.978,0.985,0.983,1.000,0.985,0.979,0.985,0.984,0.982,0.981
seed_42,0.976,0.986,0.983,0.985,1.000,0.980,0.984,0.983,0.981,0.979
seed_77,0.972,0.982,0.980,0.979,0.980,1.000,0.982,0.981,0.978,0.975
seed_100,0.976,0.984,0.983,0.985,0.984,0.982,1.000,0.983,0.981,0.981
seed_777,0.975,0.983,0.982,0.984,0.983,0.981,0.983,1.000,0.981,0.977
seed_2025,0.975,0.981,0.980,0.982,0.981,0.978,0.981,0.981,1.000,0.976
seed_9999,0.972,0.980,0.979,0.981,0.979,0.975,0.981,0.977,0.976,1.000


## **Top-k overlap (One-Class SVM)**

In [ ]:
for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(
        ocsvm_emb_scores,
        k
    )

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.813
Min overlap  : 0.600
Max overlap  : 1.000
Top-25
Mean overlap : 0.844
Min overlap  : 0.760
Max overlap  : 0.960
Top-50
Mean overlap : 0.823
Min overlap  : 0.720
Max overlap  : 0.900
Top-100
Mean overlap : 0.877
Min overlap  : 0.820
Max overlap  : 0.920


# **Combined feature space (PhysChem + ChemBERTa)**

## **Construction of the combined feature matrix**

In [ ]:
# ============================================================
# Combined feature space
# ============================================================

# PhysChem
print("PhysChem :", X_phys.shape)

# ChemBERTa
print("ChemBERTa:", X_emb.shape)

# Combined
X_combined = np.hstack([
    X_phys,
    X_emb
])

print("Combined :", X_combined.shape)

print()

print("NaN values :", np.isnan(X_combined).sum())
print("Inf values :", np.isinf(X_combined).sum())

PhysChem : (834, 183)
ChemBERTa: (834, 768)
Combined : (834, 951)

NaN values : 0
Inf values : 0


In [ ]:
print("Duplicates:", df.columns.duplicated().sum())

Duplicates: 0


## **Isolation Forest on Combined features**

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=500,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

results_if_combined = evaluate_model(
    X=X_combined,
    groups=groups,
    model=model,
    random_state=42,
)

results_if_combined["score"].describe()

,score
count,834.000000
mean,0.061164
std,0.036383
min,-0.102191
25%,0.039900
50%,0.065624
75%,0.089507
max,0.119971


In [ ]:
results_if_combined["score"].describe()

tmp = (
    results_if_combined
    .merge(
        df[["cod_id", "formula_sum", "metal_group"]],
        on="cod_id",
        how="left",
    )
)

print("\n20 most anomalous structures\n")
display(tmp.nsmallest(20, "score"))

print("\n20 most typical structures\n")
display(tmp.nlargest(20, "score"))


20 most anomalous structures



,cod_id,formula_sum_x,metal,fold,score,formula_sum_y,metal_group
576,COD6000335,C6 H4 Fe K Mn N6 O2,Fe-Mn,0,-0.102191,C6 H4 Fe K Mn N6 O2,Fe-Mn
223,COD4001132,C6 H2 Fe Mn N6 O Rb,Fe-Mn,2,-0.062702,C6 H2 Fe Mn N6 O Rb,Fe-Mn
588,COD7004034,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe,3,-0.052758,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe
382,COD4104807,C8 H4 Fe N6 Ni,Fe-Ni,2,-0.051085,C8 H4 Fe N6 Ni,Fe-Ni
573,COD4518079,C14.5 H8 Cl0 Fe2 N0 O10,Fe-Fe,2,-0.044268,C14.5 H8 Cl0 Fe2 N0 O10,Fe-Fe
666,COD7037769,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe,0,-0.043718,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe
219,COD2235024,C14 H34 Cl6 Fe2 N2 O,Fe-Fe,2,-0.042297,C14 H34 Cl6 Fe2 N2 O,Fe-Fe
156,COD1517988,C12 H12 Cs2 Fe2 N12 O6 Zn3,Fe-Zn,0,-0.040899,C12 H12 Cs2 Fe2 N12 O6 Zn3,Fe-Zn
783,COD7225178,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe,1,-0.036200,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe
488,COD4317035,C8 H14 Fe Mn N O12,Fe-Mn,2,-0.035011,C8 H14 Fe Mn N O12,Fe-Mn



20 most typical structures



,cod_id,formula_sum_x,metal,fold,score,formula_sum_y,metal_group
549,COD4346025,C12 H14 B F4 Fe2 N3 O5 S2,Fe-Fe,0,0.119971,C12 H14 B F4 Fe2 N3 O5 S2,Fe-Fe
294,COD4073173,C16 H13 Fe2 N O8 S2,Fe-Fe,1,0.118940,C16 H13 Fe2 N O8 S2,Fe-Fe
242,COD4061803,C16 H24 Fe2 N2 O6 S2,Fe-Fe,1,0.118318,C16 H24 Fe2 N2 O6 S2,Fe-Fe
550,COD4346026,C12 H14 Fe2 N4 O4 S2,Fe-Fe,3,0.118191,C12 H14 Fe2 N4 O4 S2,Fe-Fe
142,COD_2232777,C12 H11 Fe2 N2 O5 P S2,Fe-Fe,0,0.117735,C12 H11 Fe2 N2 O5 P S2,Fe-Fe
301,COD4073682,C15 H11 Fe2 N O8 S3,Fe-Fe,1,0.117723,C15 H11 Fe2 N O8 S3,Fe-Fe
760,COD7120341,C16 H9 Fe2 N O6 S2,Fe-Fe,0,0.117348,C16 H9 Fe2 N O6 S2,Fe-Fe
250,COD4062908,C16 H24 Fe2 N2 O6 S2,Fe-Fe,0,0.116983,C16 H24 Fe2 N2 O6 S2,Fe-Fe
589,COD7005027,C13 H9 Fe2 N O7 S2,Fe-Fe,1,0.116861,C13 H9 Fe2 N O7 S2,Fe-Fe
265,COD4067046,C16 H17 Fe2 O7 P S2,Fe-Fe,3,0.116789,C16 H17 Fe2 O7 P S2,Fe-Fe


### **Stability across random seeds**

In [ ]:
from sklearn.ensemble import IsolationForest

seeds = [
    1, 5, 10, 21, 42,
    77, 100, 777, 2025, 9999
]

seed_scores_if_combined = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in seeds:

    model = IsolationForest(
        n_estimators=500,
        contamination="auto",
        random_state=seed,
        n_jobs=-1,
    )

    res = evaluate_model(
        X=X_combined,
        groups=groups,
        model=model,
        random_state=seed,
    )

    seed_scores_if_combined[f"seed_{seed}"] = res["score"].values

print(seed_scores_if_combined.shape)
display(seed_scores_if_combined.head())

(834, 11)


,cod_id,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
0,1515817,0.072401,0.076267,0.068019,0.076872,0.075147,0.078643,0.069928,0.075785,0.071094,0.081541
1,1515819,0.035642,0.032225,0.039645,0.028049,0.032433,0.027214,0.026654,0.041550,0.032457,0.056188
2,1515818,0.066614,0.074796,0.070604,0.071571,0.075971,0.077772,0.062587,0.069579,0.065410,0.076942
3,1515820,0.067507,0.086572,0.070605,0.062533,0.062132,0.066185,0.060638,0.065512,0.063838,0.069462
4,1519184,0.088536,0.085966,0.084371,0.089219,0.087946,0.084629,0.093541,0.080771,0.089760,0.088777


### **Spearman correlation (Combined + Isolation Forest)**

In [ ]:
corr = seed_scores_if_combined.drop(columns="cod_id").corr(method="spearman")

display(corr)

vals = corr.values[np.triu_indices_from(corr, k=1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000000,0.982995,0.981917,0.982860,0.982247,0.982600,0.980811,0.983359,0.981573,0.981956
seed_5,0.982995,1.000000,0.982818,0.985004,0.982476,0.983463,0.982259,0.985067,0.980644,0.983487
seed_10,0.981917,0.982818,1.000000,0.983821,0.982278,0.982436,0.980201,0.982596,0.979748,0.980642
seed_21,0.982860,0.985004,0.983821,1.000000,0.985691,0.984072,0.982491,0.985006,0.982752,0.982821
seed_42,0.982247,0.982476,0.982278,0.985691,1.000000,0.984359,0.980987,0.984929,0.984604,0.983500
seed_77,0.982600,0.983463,0.982436,0.984072,0.984359,1.000000,0.982049,0.983530,0.982606,0.983653
seed_100,0.980811,0.982259,0.980201,0.982491,0.980987,0.982049,1.000000,0.982823,0.981821,0.980563
seed_777,0.983359,0.985067,0.982596,0.985006,0.984929,0.983530,0.982823,1.000000,0.983108,0.983460
seed_2025,0.981573,0.980644,0.979748,0.982752,0.984604,0.982606,0.981821,0.983108,1.000000,0.981571
seed_9999,0.981956,0.983487,0.980642,0.982821,0.983500,0.983653,0.980563,0.983460,0.981571,1.000000


Mean Spearman correlation: 0.9827
Minimum correlation      : 0.9797
Maximum correlation      : 0.9857


### **Top-k overlap (Combined + Isolation Forest)**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nlargest(k, c1).index)
        top2 = set(df_scores.nlargest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(seed_scores_if_combined, k)

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.480
Min overlap  : 0.200
Max overlap  : 0.700
Top-25
Mean overlap : 0.672
Min overlap  : 0.600
Max overlap  : 0.760
Top-50
Mean overlap : 0.790
Min overlap  : 0.660
Max overlap  : 0.880
Top-100
Mean overlap : 0.851
Min overlap  : 0.810
Max overlap  : 0.900


## **Combined + Local Outlier Factor (LOF)**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

model = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
)

results_lof_combined = evaluate_model(
    X=X_combined,
    groups=groups,
    model=model,
    random_state=42,
)

results_lof_combined["score"].describe()

,score
count,834.000000
mean,0.369824
std,0.255612
min,-5.804912
25%,0.326428
50%,0.411828
75%,0.467015
max,0.534161


In [ ]:
tmp = (
    results_lof_combined
    .merge(
        df[["cod_id", "formula_sum", "metal_group"]],
        on="cod_id",
        how="left",
    )
)

print("\n20 most anomalous structures\n")
display(tmp.nsmallest(20, "score"))

print("\n20 most typical structures\n")
display(tmp.nlargest(20, "score"))


20 most anomalous structures



,cod_id,formula_sum_x,metal,fold,score,formula_sum_y,metal_group
499,COD4322041,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe,3,-5.804912,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe
783,COD7225178,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe,1,-0.805659,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe
576,COD6000335,C6 H4 Fe K Mn N6 O2,Fe-Mn,0,-0.553835,C6 H4 Fe K Mn N6 O2,Fe-Mn
833,1514389,C50 H78 Ba F3 Fe N4 O17 S4,Fe-Ba,3,-0.524073,C50 H78 Ba F3 Fe N4 O17 S4,Fe-Ba
830,7012471,C12 H10 Ba Fe N0 O5,Fe-Ba,1,-0.393507,C12 H10 Ba Fe N0 O5,Fe-Ba
809,COD7703149,C20 H32 Cu5 Fe2 Gd N22 Na O18,Fe-Cu,1,-0.323337,C20 H32 Cu5 Fe2 Gd N22 Na O18,Fe-Cu
832,1548887,C26 H27 Ba F6 Fe N3 O12.5 S2,Fe-Ba,2,-0.320440,C26 H27 Ba F6 Fe N3 O12.5 S2,Fe-Ba
588,COD7004034,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe,3,-0.247247,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe
782,COD7225179,C10 H57 Fe2 La N6 O89 P2 W20,Fe-Fe,3,-0.242787,C10 H57 Fe2 La N6 O89 P2 W20,Fe-Fe
831,1548885,C26 H27 Ba Cl F6 Fe N3 O12 S2,Fe-Ba,0,-0.200510,C26 H27 Ba Cl F6 Fe N3 O12 S2,Fe-Ba



20 most typical structures



,cod_id,formula_sum_x,metal,fold,score,formula_sum_y,metal_group
438,COD4132746,C15 H11 Fe2 N O6 S2,Fe-Fe,2,0.534161,C15 H11 Fe2 N O6 S2,Fe-Fe
621,COD7012291,C11 H12 Fe2 O5 S3,Fe-Fe,1,0.532141,C11 H12 Fe2 O5 S3,Fe-Fe
437,COD4132744,C16 H13 Fe2 N O6 S2,Fe-Fe,2,0.528365,C16 H13 Fe2 N O6 S2,Fe-Fe
464,COD4305541,C19 H36 Fe2 N O8 P3 S2,Fe-Fe,0,0.527959,C19 H36 Fe2 N O8 P3 S2,Fe-Fe
350,COD4086200,C14 H9 Fe2 O8 P S,Fe-Fe,2,0.527545,C14 H9 Fe2 O8 P S,Fe-Fe
752,COD7114797,C10 H11 Fe2 N O5 S3,Fe-Fe,0,0.526759,C10 H11 Fe2 N O5 S3,Fe-Fe
292,COD4073171,C14 H8 Fe2 N2 O7 S2,Fe-Fe,0,0.525570,C14 H8 Fe2 N2 O7 S2,Fe-Fe
301,COD4073682,C15 H11 Fe2 N O8 S3,Fe-Fe,1,0.524560,C15 H11 Fe2 N O8 S3,Fe-Fe
697,COD7054181,C18 H33 Fe2 N O4 P2 S2,Fe-Fe,1,0.524475,C18 H33 Fe2 N O4 P2 S2,Fe-Fe
305,COD4074503,C14 H16 Fe2 O9 S2,Fe-Fe,3,0.524392,C14 H16 Fe2 O9 S2,Fe-Fe


### **Stability across random seeds (Combined + LOF)**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

seeds = [
    1, 5, 10, 21, 42,
    77, 100, 777, 2025, 9999
]

seed_scores_lof_combined = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in seeds:

    model = LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
    )

    res = evaluate_model(
        X=X_combined,
        groups=groups,
        model=model,
        random_state=seed,
    )

    seed_scores_lof_combined[f"seed_{seed}"] = res["score"].values

print(seed_scores_lof_combined.shape)
display(seed_scores_lof_combined.head())

(834, 11)


,cod_id,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
0,1515817,0.415439,0.377670,0.388149,0.401110,0.407950,0.388697,0.412614,0.382593,0.399904,0.412902
1,1515819,0.324441,0.292931,0.330015,0.333759,0.329789,0.305456,0.318408,0.260698,0.288212,0.329730
2,1515818,0.423567,0.349005,0.357910,0.354078,0.382585,0.364289,0.353009,0.382185,0.384684,0.377391
3,1515820,0.343249,0.372546,0.354743,0.333874,0.355638,0.358577,0.383695,0.381610,0.385566,0.338620
4,1519184,0.250812,0.187603,0.220813,0.189746,0.215013,0.220789,0.241080,0.240199,0.177447,0.222308


### **Spearman correlation (Combined + LOF)**

In [ ]:
corr = seed_scores_lof_combined.drop(columns="cod_id").corr(method="spearman")

display(corr)

vals = corr.values[np.triu_indices_from(corr, k=1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000000,0.961433,0.942935,0.965888,0.965134,0.952680,0.967121,0.946760,0.964698,0.954384
seed_5,0.961433,1.000000,0.946894,0.965602,0.963354,0.958689,0.961925,0.951739,0.966286,0.955506
seed_10,0.942935,0.946894,1.000000,0.946085,0.947835,0.936715,0.941027,0.920180,0.947703,0.930030
seed_21,0.965888,0.965602,0.946085,1.000000,0.967150,0.961067,0.968554,0.951642,0.968753,0.956394
seed_42,0.965134,0.963354,0.947835,0.967150,1.000000,0.959284,0.965636,0.948611,0.965631,0.951786
seed_77,0.952680,0.958689,0.936715,0.961067,0.959284,1.000000,0.954333,0.953005,0.958457,0.954762
seed_100,0.967121,0.961925,0.941027,0.968554,0.965636,0.954333,1.000000,0.947845,0.964595,0.954559
seed_777,0.946760,0.951739,0.920180,0.951642,0.948611,0.953005,0.947845,1.000000,0.947817,0.948342
seed_2025,0.964698,0.966286,0.947703,0.968753,0.965631,0.958457,0.964595,0.947817,1.000000,0.952009
seed_9999,0.954384,0.955506,0.930030,0.956394,0.951786,0.954762,0.954559,0.948342,0.952009,1.000000


Mean Spearman correlation: 0.9547
Minimum correlation      : 0.9202
Maximum correlation      : 0.9688


### **Top-k overlap (Combined + LOF)**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nlargest(k, c1).index)
        top2 = set(df_scores.nlargest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(seed_scores_lof_combined, k)

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.604
Min overlap  : 0.400
Max overlap  : 0.900
Top-25
Mean overlap : 0.726
Min overlap  : 0.640
Max overlap  : 0.840
Top-50
Mean overlap : 0.803
Min overlap  : 0.720
Max overlap  : 0.880
Top-100
Mean overlap : 0.867
Min overlap  : 0.810
Max overlap  : 0.920


## **Combined + One-Class SVM**

In [ ]:
from sklearn.svm import OneClassSVM

model = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05,
)

results_svm_combined = evaluate_model(
    X=X_combined,
    groups=groups,
    model=model,
    random_state=42,
)

results_svm_combined["score"].describe()

,score
count,834.000000
mean,0.458165
std,0.429644
min,-2.105489
25%,0.227356
50%,0.508833
75%,0.735379
max,1.490482


In [ ]:
tmp = (
    results_svm_combined
    .merge(
        df[["cod_id", "formula_sum", "metal_group"]],
        on="cod_id",
        how="left",
    )
)

print("\n20 most anomalous structures\n")
display(tmp.nsmallest(20, "score"))

print("\n20 most typical structures\n")
display(tmp.nlargest(20, "score"))


20 most anomalous structures



,cod_id,formula_sum_x,metal,fold,score,formula_sum_y,metal_group
499,COD4322041,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe,3,-2.105489,C0 H72 Fe2 K6 Na O105 Sb2 W20,Fe-Fe
783,COD7225178,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe,1,-1.899908,C10 H53 Ce Fe2 N6 O87 P2 W20,Fe-Fe
576,COD6000335,C6 H4 Fe K Mn N6 O2,Fe-Mn,0,-1.659430,C6 H4 Fe K Mn N6 O2,Fe-Mn
809,COD7703149,C20 H32 Cu5 Fe2 Gd N22 Na O18,Fe-Cu,1,-1.297907,C20 H32 Cu5 Fe2 Gd N22 Na O18,Fe-Cu
830,7012471,C12 H10 Ba Fe N0 O5,Fe-Ba,1,-1.128606,C12 H10 Ba Fe N0 O5,Fe-Ba
666,COD7037769,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe,0,-1.009087,C16 H18 Br6 Fe2 In2 N6 O4,Fe-Fe
833,1514389,C50 H78 Ba F3 Fe N4 O17 S4,Fe-Ba,3,-0.985482,C50 H78 Ba F3 Fe N4 O17 S4,Fe-Ba
219,COD2235024,C14 H34 Cl6 Fe2 N2 O,Fe-Fe,2,-0.961928,C14 H34 Cl6 Fe2 N2 O,Fe-Fe
668,COD7037888,C12 H60 Fe2 N8 Na2 O88 Sb2 W20,Fe-Fe,3,-0.911851,C12 H60 Fe2 N8 Na2 O88 Sb2 W20,Fe-Fe
588,COD7004034,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe,3,-0.853338,C10 H75 Fe2 K8 N5 O91 Si2 W20,Fe-Fe



20 most typical structures



,cod_id,formula_sum_x,metal,fold,score,formula_sum_y,metal_group
550,COD4346026,C12 H14 Fe2 N4 O4 S2,Fe-Fe,3,1.490482,C12 H14 Fe2 N4 O4 S2,Fe-Fe
549,COD4346025,C12 H14 B F4 Fe2 N3 O5 S2,Fe-Fe,0,1.441605,C12 H14 B F4 Fe2 N3 O5 S2,Fe-Fe
628,COD7015307,C14 H21 Fe2 I O5 S3,Fe-Fe,3,1.342348,C14 H21 Fe2 I O5 S3,Fe-Fe
817,COD7709297,C18 H18 Fe2 N2 O2,Fe-Fe,3,1.293191,C18 H18 Fe2 N2 O2,Fe-Fe
795,COD7236139,C18 H21 Fe2 N2 O14 S2,Fe-Fe,2,1.263208,C18 H21 Fe2 N2 O14 S2,Fe-Fe
775,COD7219887,C18 H22 Fe2 N4 O4 S2,Fe-Fe,0,1.260923,C18 H22 Fe2 N4 O4 S2,Fe-Fe
423,COD4122587,C10 H13 Fe2 O5 P S2,Fe-Fe,2,1.259039,C10 H13 Fe2 O5 P S2,Fe-Fe
280,COD4070706,C10 H9 Fe2 N O6 S2,Fe-Fe,3,1.247844,C10 H9 Fe2 N O6 S2,Fe-Fe
295,COD4073238,C19 H27 Fe2 N O3 Si3,Fe-Fe,0,1.241778,C19 H27 Fe2 N O3 Si3,Fe-Fe
446,COD4304478,C14 H8 Fe2 O8 S3,Fe-Fe,1,1.235768,C14 H8 Fe2 O8 S3,Fe-Fe


In [ ]:
from sklearn.svm import OneClassSVM

seeds = [
    1, 5, 10, 21, 42,
    77, 100, 777, 2025, 9999
]

seed_scores_svm_combined = pd.DataFrame({
    "cod_id": df["cod_id"].values
})

for seed in seeds:

    model = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
    )

    res = evaluate_model(
        X=X_combined,
        groups=groups,
        model=model,
        random_state=seed,
    )

    seed_scores_svm_combined[f"seed_{seed}"] = res["score"].values

print(seed_scores_svm_combined.shape)
display(seed_scores_svm_combined.head())

(834, 11)


,cod_id,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
0,1515817,0.512255,0.542019,0.595731,0.628190,0.656289,0.652593,0.573841,0.646228,0.522038,0.602713
1,1515819,-0.304176,-0.272435,-0.164009,-0.171397,-0.203797,-0.181885,-0.177056,-0.111963,-0.243912,-0.127872
2,1515818,0.562628,0.448580,0.459641,0.487875,0.504412,0.440533,0.445563,0.537791,0.636807,0.604950
3,1515820,0.585075,0.632911,0.644160,0.469322,0.577049,0.397810,0.613360,0.627078,0.410947,0.326218
4,1519184,0.501569,0.481992,0.493437,0.510656,0.491696,0.507283,0.487817,0.439013,0.538426,0.483553


### **Spearman correlation**

In [ ]:
corr = seed_scores_svm_combined.drop(columns="cod_id").corr(method="spearman")

display(corr)

vals = corr.values[np.triu_indices_from(corr, k=1)]

print("=" * 60)
print(f"Mean Spearman correlation: {vals.mean():.4f}")
print(f"Minimum correlation      : {vals.min():.4f}")
print(f"Maximum correlation      : {vals.max():.4f}")

,seed_1,seed_5,seed_10,seed_21,seed_42,seed_77,seed_100,seed_777,seed_2025,seed_9999
seed_1,1.000000,0.968723,0.970377,0.973648,0.971429,0.971047,0.969241,0.972154,0.971287,0.966224
seed_5,0.968723,1.000000,0.978455,0.981677,0.978130,0.977072,0.978342,0.976657,0.978191,0.972189
seed_10,0.970377,0.978455,1.000000,0.983851,0.980691,0.978914,0.978204,0.979042,0.979512,0.975731
seed_21,0.973648,0.981677,0.983851,1.000000,0.983483,0.981702,0.983802,0.983054,0.982808,0.981027
seed_42,0.971429,0.978130,0.980691,0.983483,1.000000,0.978276,0.978522,0.979241,0.978455,0.974776
seed_77,0.971047,0.977072,0.978914,0.981702,0.978276,1.000000,0.977124,0.978774,0.979449,0.974123
seed_100,0.969241,0.978342,0.978204,0.983802,0.978522,0.977124,1.000000,0.977933,0.979155,0.975577
seed_777,0.972154,0.976657,0.979042,0.983054,0.979241,0.978774,0.977933,1.000000,0.979498,0.974034
seed_2025,0.971287,0.978191,0.979512,0.982808,0.978455,0.979449,0.979155,0.979498,1.000000,0.977880
seed_9999,0.966224,0.972189,0.975731,0.981027,0.974776,0.974123,0.975577,0.974034,0.977880,1.000000


Mean Spearman correlation: 0.9771
Minimum correlation      : 0.9662
Maximum correlation      : 0.9839


### **Top-k overlap**

In [ ]:
from itertools import combinations

def top_overlap(df_scores, k):

    cols = list(df_scores.columns[1:])

    overlaps = []

    for c1, c2 in combinations(cols, 2):

        top1 = set(df_scores.nlargest(k, c1).index)
        top2 = set(df_scores.nlargest(k, c2).index)

        overlaps.append(len(top1 & top2) / k)

    return np.mean(overlaps), np.min(overlaps), np.max(overlaps)


for k in [10, 25, 50, 100]:

    mean_, min_, max_ = top_overlap(seed_scores_svm_combined, k)

    print("=" * 60)
    print(f"Top-{k}")
    print(f"Mean overlap : {mean_:.3f}")
    print(f"Min overlap  : {min_:.3f}")
    print(f"Max overlap  : {max_:.3f}")

Top-10
Mean overlap : 0.631
Min overlap  : 0.400
Max overlap  : 0.800
Top-25
Mean overlap : 0.786
Min overlap  : 0.680
Max overlap  : 0.880
Top-50
Mean overlap : 0.811
Min overlap  : 0.740
Max overlap  : 0.860
Top-100
Mean overlap : 0.856
Min overlap  : 0.790
Max overlap  : 0.900


# **SAVED**

PHYS

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib
from pathlib import Path
Path("../results/models").mkdir(parents=True, exist_ok=True)

# Final pipeline
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", IsolationForest(
        random_state=42,
        contamination="auto"
    ))
])

pipe.fit(X_phys)

joblib.dump(pipe, "../results/models/PhysChem_IsolationForest.pkl")

print("Saved.")

Saved.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import joblib

# Final pipeline for ChemBERTa
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", IsolationForest(
        random_state=42,
        contamination="auto"
    ))
])

# Fit on the full ChemBERTa feature set
pipe.fit(X_emb)

# Save
joblib.dump(pipe, "../results/models/ChemBERTa_IsolationForest.pkl")

print("ChemBERTa Isolation Forest model saved.")

ChemBERTa Isolation Forest model saved.


In [ ]:
# ============================================================
# Save all results
# ============================================================

from pathlib import Path

OUT_DIR = Path("../results/representation_benchmark")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(OUT_DIR.resolve())

/content/results


In [ ]:
print(sorted([x for x in globals().keys() if "result" in x.lower()]))

['results_if_combined', 'results_lof_combined', 'results_svm_combined']


In [ ]:
# ============================================================
# Save final anomaly rankings
# ============================================================

from pathlib import Path

OUT_DIR = Path("../results/representation_benchmark")
OUT_DIR.mkdir(parents=True, exist_ok=True)

results_if_combined.to_csv(
    OUT_DIR / "chemberta_if_scores.csv",
    index=False,
)

results_lof_combined.to_csv(
    OUT_DIR / "chemberta_lof_scores.csv",
    index=False,
)

results_svm_combined.to_csv(
    OUT_DIR / "chemberta_ocsvm_scores.csv",
    index=False,
)

print("Saved:")
for f in OUT_DIR.iterdir():
    print("  ", f.name)

Saved:
   chemberta_if_scores.csv
   chemberta_ocsvm_scores.csv
   chemberta_lof_scores.csv


In [ ]:
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name:30s} {obj.shape}")

features                       (990, 190)
emb                            (970, 770)
df                             (970, 960)
results_if_combined            (970, 5)
tmp                            (970, 7)
seed_scores_if_combined        (970, 11)
res                            (970, 5)
corr                           (10, 10)
results_lof_combined           (970, 5)
seed_scores_lof_combined       (970, 11)
results_svm_combined           (970, 5)
seed_scores_svm_combined       (970, 11)


In [ ]:
config = """
Dataset: Combined
Objects: 834

PhysChem features: 183
ChemBERTa features: 768
Total features: 951

Algorithms:
- Isolation Forest (500 trees)
- Local Outlier Factor
- One-Class SVM

Seeds:
1,5,10,21,42,77,100,777,2025,9999

Final model used for ranking:
Isolation Forest
"""

with open(OUT_DIR / "experiment_config.txt", "w") as f:
    f.write(config)

print("Configuration saved.")

Configuration saved.


In [ ]:
# ============================================================
# Save final dataset
# ============================================================

df.to_csv(
    OUT_DIR / "combined_dataset.csv",
    index=False,
)

print("Saved combined_dataset.csv")

Saved combined_dataset.csv


In [ ]:
seed_scores_if_combined.to_csv(
    OUT_DIR / "if_all_seeds.csv",
    index=False,
)

seed_scores_lof_combined.to_csv(
    OUT_DIR / "lof_all_seeds.csv",
    index=False,
)

seed_scores_svm_combined.to_csv(
    OUT_DIR / "ocsvm_all_seeds.csv",
    index=False,
)